# Phase 1: Setup & Data Loading
Run this notebook to download the Ireland hourly weather dataset using the Kaggle API.

In [1]:
import os
import sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('GPNCM'):
        !git clone https://github.com/TinevimboMusingadi/GPNCM.git
    %cd GPNCM
    PROJECT_DIR = '/content/GPNCM'
    print("Running in Colab, working directory is now GPNCM")
else:
    PROJECT_DIR = '..'
    print("Running locally")

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)


/content/GPNCM
Running in Colab, working directory is now GPNCM


In [2]:
!pip install kagglehub pandas pyarrow -q

import kagglehub
import pandas as pd
import glob
import os

project_dir = PROJECT_DIR
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)

# Download the Ireland hourly weather dataset
path = kagglehub.dataset_download("dariasvasileva/hourly-weather-data-in-ireland-from-24-stations")
print("Dataset downloaded to:", path)

# Merge all CSVs if multiple exist, or just copy the main one
files = glob.glob(f'{path}/**/*.csv', recursive=True)
dfs = []
for f in files:
    try:
        d = pd.read_csv(f)
        if 'station' not in d.columns:
            d['station'] = os.path.basename(f).replace('.csv', '')
        dfs.append(d)
    except Exception as e:
        print(f"Could not read {f}: {e}")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    df_all.to_parquet(f'{PROJECT_DIR}/data/ireland_all_stations.parquet', index=False)
    print(f"Merged data saved. Total rows: {len(df_all):,}")

Using Colab cache for faster access to the 'hourly-weather-data-in-ireland-from-24-stations' dataset.
Dataset downloaded to: /kaggle/input/hourly-weather-data-in-ireland-from-24-stations
Merged data saved. Total rows: 7,856,898


# Phase 2: Data Cleaning & Exploratory Data Analysis

In [3]:
import sys
sys.path.append(PROJECT_DIR)

import pandas as pd
import yaml
from utils.plot_utils import plot_feature_distributions, plot_correlation_matrix
from src.data_loader import load_data

# Load configuration
with open(f'{PROJECT_DIR}/config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

feature_cols = config['data']['feature_cols']

# Load merged data
df = pd.read_parquet(f'{PROJECT_DIR}/data/ireland_all_stations.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Handle Missing Values
df[feature_cols] = df[feature_cols].fillna(method='ffill', limit=3)
df[feature_cols] = df[feature_cols].fillna(method='bfill', limit=3)

# Drop rows still missing the target variable
df = df.dropna(subset=[config['data']['target_col']])

# Cap extreme outliers for rain
df['rain'] = df['rain'].clip(lower=0, upper=100)

# Save cleaned dataset
df[['date'] + feature_cols].to_parquet(f'{PROJECT_DIR}/data/ireland_clean.parquet', index=False)
print(f"Clean dataset saved: {len(df):,} rows")


: 

: 

: 

In [ ]:
# Plotting EDA
plot_feature_distributions(df, feature_cols, f'{PROJECT_DIR}/outputs/feature_distributions.png')
plot_correlation_matrix(df, feature_cols, f'{PROJECT_DIR}/outputs/correlation_matrix.png')